# Algorytmika i matematyka uczenia maszynowego 
## Laboratorium 11

### Zadanie 1

Zaimplementuj systemu rekomendacji filmów w oparciu o indeks Jaccarda. System ma za zadanie zwrócić listę filmów sugerowany dla podanego użytownika.

Dane zostały pobrane z serwisu Kaggle z https://www.kaggle.com/datasets/gargmanas/movierecommenderdataset

Zbiór zawiera dwa pliki:
- `movies.csv` lista filmów wraz z ich identyfikatorami
- `ratings.csv` lista ocen filmów przez użytkowników

**Zadanie:**
* Wczytaj oba pliki.
* Zamień wszystkie oceny użytkownika na wartość 1 (zastosuj próg okreśjący czy film się podobał czy nie np. 3).
* Stwórz macierz ocen użytkowników w której wierszach będą użytkownicy, a w kolumnach filmy. Wartość w macierzy jest flagą mówiącą czy użytkownikowi film się podobał czy nie. 
* Wypełnij brakujące wartości zerami.
* Utwórz macierz podobieństwa Jaccarda pomiędzy użytkownikami (każdy z każdym).
    - Możesz wykorzystać funkcję [jaccard](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.jaccard.html) z biblioteki scipy.
* Zaimplementuj funkcję która dla podanego użytkownika zwróci listę sugerowanych filmów.
    - Funkcja powinna zwrócić listę filmów które nie były ocenione przez użytkownika, a które są rekomendowane dla niego.



In [11]:
import pandas as pd
from scipy.spatial.distance import pdist, squareform, jaccard

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

ratings['rating'] = (ratings['rating'] > 2.5).astype('int')
print(ratings['rating'].unique())

user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating', fill_value=0)

jaccard_distances = pdist(user_movie_matrix.values, metric='jaccard')
jaccard_similarity_matrix = 1 - squareform(jaccard_distances)

print(movies.head())
print(ratings.head())

[1 0]
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1       1  964982703
1       1        3       1  964981247
2       1        6       1  964982224
3       1       47       1  964983815
4       1       50       1  964982931




### Zadanie 2


Algorytm MinHash na przykładzie wykrywania plagiatów

Wykonaj kolejno następujące kroki:

1. Pobierz 8 akapitów tekstu (nie za krótkich), każdy o różnej tematyce (mogą być np. z różnych haseł Wikipedii), trzymaj się jednego języka (np. PL lub ENG). Wklej je do jednego pliku tekstowego, z linią wolną jako separatorem.

2. Skopiuj wybrane 2-3 akapity i ręcznie nieco zmodyfikuj.

> Przykład (z hasła https://pl.wikipedia.org/wiki/Fryderyk_Chopin):

```Jest uważany za jednego z najwybitniejszych kompozytorów romantycznych, a także za jednego z najważniejszych polskich kompozytorów w historii. Był jednym z najsłynniejszych pianistów swoich czasów, często nazywany poetą fortepianu. Elementem charakterystycznym dla utworów Chopina jest pogłębiona ekspresja oraz czerpanie z wzorców stylistycznych polskiej muzyki ludowej.```

↓↓↓ Zmieniono na ↓↓↓

```Jest uważany za jednego z najwybitniejszych kompozytorów romantycznych, a także za jednego z najważniejszych kompozytorów polskich w historii.  Był jednym  z najsłynniejszych pianistów swoich czasów, często nazywany poetą fortepianu! Elementem charakterystycznym dla utworów Fryderyka Chopina jest pogłębiona ekspresja oraz czerpanie z wzorców polskiej muzyki ludowej.```

Otrzymasz zatem w pliku tekstowym 10 lub 11 akapitów tekstu (kolejność dowolna, te „splagiatowane” nie muszą być na końcu).

3. Z poziomu skryptu: wczytaj wszystkie akapity z pliku. Zbuduj 100 "losowych" funkcji haszujących.

> Sugestia: funkcją "bazową" jest po prostu `hash(...)`. Zakładamy 64-bitową wersję Pythona 3.x, wtedy `hash(...)` jest 64-bitowy.

Na liście seeds umieszczamy 100 losowych liczb 64-bitowych. Aby obliczyć $i$-ty hash dla ciągu 
należy wykonać `hash(s) ^ seeds[i]` (użycie operatora XOR).

4. Przyjmij niewielką wartość $Q$ (np. 15) i dla każdego akapitu
    - oznacz jego długość przez $n$,
    - dla każdej ze 100 funkcji haszujących policz hasza w przesuwnym oknie tekstu o długości $Q$ znaków (czyli łącznie mamy $n - Q + 1$ wartości hasza); zapamiętaj MINIMUM z tych $n - Q + 1$
 wartości.
Na wyjściu mamy zatem (dla 11 akapitów) 11 * 100 wartości haszy.

5. Rozważ pary akapitów "każdy z każdym". Jeśli dla danej pary co najmniej (np.) 30 haszy jest wspólnych, to uważamy akapity za podobne (być może plagiat) i wyświetlamy na ekranie.

6. Wyświetl czas obliczeń (powinien wynosić mniej niż 0.5s).

7. Poeksperymentuj z liczbą użytych funkcji haszujących, wartością, stopniem modyfikacji oryginalnych akapitów tekstu, progiem detekcji akapitów podobnych.

In [ ]:
# UZUPEŁNIJ